# Garmin Health Data Backfill -- Direct Delta Write

Backfills historical health data from Garmin Connect for a date range and
writes directly to the `wearables_zerobus` bronze Delta table using Spark SQL
with `parse_json()`. No ZeroBus dependency.

**Rate limiting:** Garmin Connect may throttle requests if too many are made
in quick succession. This notebook includes a configurable delay between dates.

**Prerequisites:**
- The `wearables_zerobus` table must exist (run notebook 00 or deploy the infra bundle).
- Garmin OAuth tokens must be stored in the secret scope (run notebook 00 first).

In [ ]:
%pip install garminconnect>=0.2.0

## Configuration

In [ ]:
DEFAULT_CATALOG = ""  # Must be supplied via widget or ${var.catalog} job parameter
DEFAULT_SCHEMA = "wearables"
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_DEVICE_ID = "garmin_forerunner_265"
DEFAULT_DELAY_SECONDS = "2"

try:
    dbutils.widgets.text("catalog", DEFAULT_CATALOG, "Catalog")
    dbutils.widgets.text("schema", DEFAULT_SCHEMA, "Schema")
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "Secret Scope")
    dbutils.widgets.text("start_date", "", "Start Date (YYYY-MM-DD, inclusive)")
    dbutils.widgets.text("end_date", "", "End Date (YYYY-MM-DD, exclusive)")
    dbutils.widgets.text("device_id", DEFAULT_DEVICE_ID, "Device ID")
    dbutils.widgets.text("delay_seconds", DEFAULT_DELAY_SECONDS, "Delay between dates (seconds)")

    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    secret_scope = dbutils.widgets.get("secret_scope")
    start_date_str = dbutils.widgets.get("start_date")
    end_date_str = dbutils.widgets.get("end_date")
    device_id = dbutils.widgets.get("device_id")
    delay_seconds = int(dbutils.widgets.get("delay_seconds"))
except Exception:
    catalog = DEFAULT_CATALOG
    schema = DEFAULT_SCHEMA
    secret_scope = DEFAULT_SECRET_SCOPE
    start_date_str = ""
    end_date_str = ""
    device_id = DEFAULT_DEVICE_ID
    delay_seconds = int(DEFAULT_DELAY_SECONDS)

spark.sql(f"USE CATALOG {catalog}")

bronze_table = f"{catalog}.{schema}.wearables_zerobus"

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Bronze table: {bronze_table}")
print(f"Date range:   {start_date_str} to {end_date_str}")
print(f"Delay:        {delay_seconds}s between dates")

## Validate Date Range

In [ ]:
from datetime import date, timedelta

if not start_date_str or not end_date_str:
    raise ValueError("Both start_date and end_date must be provided for backfill")

start_date = date.fromisoformat(start_date_str.strip())
end_date = date.fromisoformat(end_date_str.strip())

if start_date >= end_date:
    raise ValueError(f"start_date ({start_date}) must be before end_date ({end_date})")

num_days = (end_date - start_date).days
print(f"Will backfill {num_days} days: {start_date} to {end_date - timedelta(days=1)}")

## Load Garmin Tokens and Authenticate

In [ ]:
from pathlib import Path

garmin_tokens_json = dbutils.secrets.get(scope=secret_scope, key="garmin_oauth_tokens")

tokenstore = Path.home() / ".garminconnect"
tokenstore.mkdir(parents=True, exist_ok=True)
token_path = tokenstore / "garmin_tokens.json"
token_path.write_text(garmin_tokens_json)

from garminconnect import Garmin

garmin_client = Garmin()
garmin_client.login(str(tokenstore))
print("Garmin: authenticated")

## Extraction and Bronze Row Builder

In [ ]:
import json
import uuid
from datetime import datetime, timezone

EXTRACTORS = [
    ("daily_stats", "get_stats"),
    ("heart_rates", "get_heart_rates"),
    ("sleep", "get_sleep_data"),
    ("stress", "get_stress_data"),
    ("hrv", "get_hrv_data"),
    ("spo2", "get_spo2_data"),
    ("body_battery", "get_body_battery"),
    ("steps", "get_steps_data"),
    ("respiration", "get_respiration_data"),
]


def to_bronze_row(raw_data, record_type, dev_id, target_ds):
    now = datetime.now(timezone.utc).isoformat()
    body = {
        "source": "garmin_connect",
        "device_id": dev_id,
        "date": target_ds,
        "data": raw_data,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Platform": "garmin_connect",
        "X-Record-Type": record_type,
        "X-Device-Id": dev_id,
        "X-Upload-Timestamp": now,
    }
    return {
        "record_id": str(uuid.uuid4()),
        "ingested_at": now,
        "body_json": json.dumps(body, default=str),
        "headers_json": json.dumps(headers),
        "record_type": record_type,
    }


def extract_day(client, target_date, dev_id):
    ds = target_date.isoformat()
    rows = []
    for record_type, method_name in EXTRACTORS:
        try:
            raw = getattr(client, method_name)(ds)
            if raw is not None:
                rows.append(to_bronze_row(raw, record_type, dev_id, ds))
        except Exception:
            pass
    try:
        activities = client.get_activities_fordate(ds)
        if activities is not None:
            rows.append(to_bronze_row(activities, "activities", dev_id, ds))
    except Exception:
        pass
    return rows


print("Extraction helpers loaded")

## Backfill Loop

Iterates over each date, extracts data, and writes a batch of bronze rows
to the Delta table using `parse_json()`.

In [ ]:
import time
from pyspark.sql import Row
from pyspark.sql.types import StringType, StructField, StructType

staging_schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("body_json", StringType(), False),
    StructField("headers_json", StringType(), False),
    StructField("record_type", StringType(), True),
])

total_rows = 0
failed_dates = []
current = start_date

while current < end_date:
    ds = current.isoformat()
    print(f"\n--- {ds} ---")

    try:
        rows = extract_day(garmin_client, current, device_id)
        print(f"  Extracted {len(rows)} record types")

        if rows:
            spark_rows = [Row(**r) for r in rows]
            staging_df = spark.createDataFrame(spark_rows, schema=staging_schema)
            staging_df.createOrReplaceTempView("garmin_backfill_staging")

            spark.sql(f"""
                INSERT INTO {bronze_table} (record_id, ingested_at, body, headers, record_type)
                SELECT
                    record_id,
                    CAST(ingested_at AS TIMESTAMP),
                    parse_json(body_json),
                    parse_json(headers_json),
                    record_type
                FROM garmin_backfill_staging
            """)
            total_rows += len(rows)
            print(f"  Wrote {len(rows)} rows to {bronze_table}")

    except Exception as e:
        print(f"  FAILED: {e}")
        failed_dates.append(ds)

    current += timedelta(days=1)

    if current < end_date:
        time.sleep(delay_seconds)

print(f"\n=== Backfill complete ===")
print(f"Total rows written: {total_rows}")
if failed_dates:
    print(f"Failed dates ({len(failed_dates)}): {', '.join(failed_dates)}")
else:
    print("No failures")

## Save Refreshed Tokens

In [ ]:
refreshed_tokens = token_path.read_text()

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret(scope=secret_scope, key="garmin_oauth_tokens", string_value=refreshed_tokens)
print(f"Refreshed tokens saved back to scope '{secret_scope}'")

## Verify

In [ ]:
print(f"=== {bronze_table} ===")
print()

total_df = spark.sql(f"""
    SELECT COUNT(*) as total_rows
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
""")
display(total_df)

print("\n--- Record types ---")
types_df = spark.sql(f"""
    SELECT
        record_type,
        COUNT(*) as cnt,
        MIN(body:date::STRING) as earliest_date,
        MAX(body:date::STRING) as latest_date
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
    GROUP BY record_type
    ORDER BY cnt DESC
""")
display(types_df)

print(f"\nBackfill verification complete.")